# Corporate Health & Risk Radar — Exploratory Analysis
Quick-start notebook to explore the models interactively.

In [ ]:
import sys
sys.path.insert(0, '../src')

from utils.data_fetcher import fetch_multiple
from models.health_score import calculate_health_score
from models.risk_radar import calculate_risk_score
import pandas as pd

WATCHLIST = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'META', 'AMZN', 'GOOGL', 'JPM']

print('Fetching data...')
raw = fetch_multiple(WATCHLIST)
print(f'Fetched {len(raw)} companies')

In [ ]:
# Build summary dataframe
rows = []
for ticker, metrics in raw.items():
    h  = calculate_health_score(metrics)
    ri = calculate_risk_score(metrics)
    rows.append({
        'Ticker':       ticker,
        'Company':      metrics.get('name', ticker),
        'Sector':       metrics.get('sector', 'Unknown'),
        'Health Score': h.total_score,
        'Grade':        h.grade,
        'Risk Signals': ri.risk_score,
        'Risk Level':   ri.risk_level,
        'Altman Z':     ri.altman_z_proxy,
        'Rev Growth':   metrics.get('revenueGrowth'),
        'Net Margin':   metrics.get('profitMargins'),
        'ROA':          metrics.get('returnOnAssets'),
        'D/E':          metrics.get('debtToEquity'),
    })

df = pd.DataFrame(rows).sort_values('Health Score', ascending=False)
df

In [ ]:
# Deep dive on a single company
ticker = 'AAPL'
h  = calculate_health_score(raw[ticker])
ri = calculate_risk_score(raw[ticker])

print(f'=== {ticker} ===')
print(f'Health Score: {h.total_score:.1f} ({h.grade})')
print(f'Interpretation: {h.interpretation}')
print()
print('Components:')
for k, v in h.components.items():
    print(f'  {k:<16} {v:.1f}')
print()
print(f'Risk Level: {ri.risk_level} ({ri.risk_score} signals)')
print(f'Altman Z (proxy): {ri.altman_z_proxy}')
print()
print('Triggered signals:')
for s in ri.signals:
    if s.triggered:
        print(f'  [{s.severity}] {s.name}: {s.description}')